In [1]:
import csv
import math
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

from utils import (
    read_intensities, 
    read_unique_kmer_positions,
    read_kmer_positions,
    get_all_kmer_wildcards,
    get_sequence_from_fasta,
    match_all_kmers_to_wildcards,
    regression_driver,
    print_rows_as_tsv,
    plot_aff_motif_effects,
    extract_covariates
)

genome =  "../../../../../d/OneDrive - McGill University/repos/hg38_ucsc.fa"
intensities = read_intensities("data/mcf7_gabpa/GABPA_MCF7_probeIntensity.bed")
kmers = read_unique_kmer_positions("data/mcf7_gabpa/GABPA_Array_ATAC.txt")
dhs = 1
# region = "chr5:1295105-1295140"
# region = "chr6:1295220-1295228"
num_random = 1000
mode = "neg-binomial"

/home/aki/miniconda3/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
kmer_size = 8
# snv_list =  "chr5:1295113:G>A"
snv_list =  "chr5:1295113:C>T"
# snv_list =  "chr5:1295113:C>T,chr5:1295135:C>T"

In [3]:
import sys
import re

def parse_snv_string(snv_str):
    chrom, pos, change = snv_str.split(":")
    ref, alt = change.split(">")
    return chrom, int(pos), ref.upper(), alt.upper()

def reverse_complement(seq):
    complement = str.maketrans("ACGTacgt", "TGCAtgca")
    return seq.translate(complement)[::-1]

def normalize_snv_region(chrom, snv_pos, ref_allele, alt_allele, genome, kmer_size, debug=False):
    """
    Normalize SNV region to a window of size 2*kmer_size - 1, centered on the SNV.
    Automatically reverse complements if the ref allele doesn't match the genome.
    """
    half_window = kmer_size - 1
    region_start = snv_pos - half_window
    region_end = snv_pos + half_window  # exclusive

    region_seq = get_sequence_from_fasta(chrom, region_start, region_end, genome)
    snv_index = snv_pos - region_start
    genome_base = region_seq[snv_index]

    if debug:
        sys.stderr.write(f"\nWindowed region: {chrom}:{region_start}-{region_end}\n")
        sys.stderr.write(f"Sequence: {region_seq}\n")
        sys.stderr.write(f"SNV should be at position {snv_index} → base: {genome_base}\n")

    if genome_base == ref_allele:
        if debug:
            sys.stderr.write("[*] Reference allele matches genome — no reverse complement needed.\n")
    else:
        if debug:
            sys.stderr.write(f"[!] Reference allele mismatch: genome has '{genome_base}', SNV claims '{ref_allele}'\n")
            sys.stderr.write("[*] Trying reverse complement...\n")

        region_seq = reverse_complement(region_seq)
        snv_index = len(region_seq) - 1 - snv_index
        
        genome_base = region_seq[snv_index]

        if debug:
            sys.stderr.write(f"[debug] Reverse complemented sequence: {region_seq}\n")
            sys.stderr.write(f"[debug] Reverse complemented snv_index: {snv_index}\n")
            sys.stderr.write(f"[debug] Reverse complemented genome_base: {genome_base}\n")
            sys.stderr.write(f"[debug] Reverse complemented ref_allele: {ref_allele}\n")
            sys.stderr.write(f"[debug] reverse_complement('C') = {reverse_complement('C')}\n")

        if genome_base != ref_allele:
            raise ValueError(
                f"[ERROR] Reference allele mismatch even after reverse complement. "
                f"Genome has '{genome_base}', expected '{ref_allele}' at index {snv_index}"
            )

        if debug:
            sys.stderr.write("[*] Using reverse complement:\n")
            sys.stderr.write(f"    Updated sequence: {region_seq}\n")
            sys.stderr.write(f"    Updated ref: {ref_allele}, alt: {alt_allele}\n")

    # Final debug before returning
    if debug:
        sys.stderr.write("\n[returning]\n")
        sys.stderr.write(f"  region_seq: {region_seq}\n")
        sys.stderr.write(f"  snv_index: {snv_index}\n")
        sys.stderr.write(f"  ref_allele: {ref_allele}\n")
        sys.stderr.write(f"  genome_base (final): {region_seq[snv_index]}\n")

    return {
        "chrom": chrom,
        "pos": snv_pos,
        "ref": ref_allele,
        "alt": alt_allele,
        "region_start": region_start,
        "region_end": region_end,
        "region_seq": region_seq,
        "snv_index": snv_index,
    }


parsed_snvs = []

for snv_str in snv_list.split(","):
    chrom, snv_pos, ref_allele, alt_allele = parse_snv_string(snv_str)
    print(f"\nParsed SNV: {chrom}, {snv_pos}, {ref_allele} > {alt_allele}")
    
    snv_info = normalize_snv_region(chrom, snv_pos, ref_allele, alt_allele, genome, kmer_size, debug = True)
    snv_info["snv_str"] = snv_str
    parsed_snvs.append(snv_info)


Parsed SNV: chr5, 1295113, C > T



Windowed region: chr5:1295106-1295120
Sequence: GCCCGGAGGGGGCTG
SNV should be at position 7 → base: G
[!] Reference allele mismatch: genome has 'G', SNV claims 'C'
[*] Trying reverse complement...
[debug] Reverse complemented sequence: CAGCCCCCTCCGGGC
[debug] Reverse complemented snv_index: 7
[debug] Reverse complemented genome_base: C
[debug] Reverse complemented ref_allele: C
[debug] reverse_complement('C') = G
[*] Using reverse complement:
    Updated sequence: CAGCCCCCTCCGGGC
    Updated ref: C, alt: T

[returning]
  region_seq: CAGCCCCCTCCGGGC
  snv_index: 7
  ref_allele: C
  genome_base (final): C


In [4]:
def get_snv_aligned_wildcards(snv_info, kmer_size):
    wildcards = []
    for start in range(len(snv_info["region_seq"]) - kmer_size + 1):
        end = start + kmer_size
        if start <= snv_info["snv_index"] < end:
            rel_pos = snv_info["snv_index"] - start
            kmer = snv_info["region_seq"][start:end]
            wildcard = kmer[:rel_pos] + "." + kmer[rel_pos + 1:]
            wildcards.append((start, wildcard))
    return wildcards


def match_snv_aligned_kmers(snv_info, kmer_positions, kmer_size, debug=False):
    """
    Match k-mers from kmer_positions to SNV-aligned wildcards.
    Returns:
        allele_region_offsets: {motif_pos: {allele: {region: offset}}}
        allele_matched_kmers: {motif_pos: {allele: [matched_kmers]}}
    """
    allele_region_offsets = {}
    allele_matched_kmers = {}

    wildcards = get_snv_aligned_wildcards(snv_info, kmer_size)

    for motif_pos, wildcard in wildcards:
        pattern = re.compile("^" + wildcard.replace(".", "[ACGT]") + "$")
        snv_index = wildcard.index(".")

        # Temporary: region → alleles map for deduplication
        region_to_alleles = {}
        region_kmer_hits = {}

        # First pass: collect all matches
        for kmer, region_dict in kmer_positions.items():
            if pattern.fullmatch(kmer):
                allele = kmer[snv_index]

                for region_id, offset in region_dict.items():
                    region_to_alleles.setdefault(region_id, set()).add(allele)
                    region_kmer_hits.setdefault(region_id, {})[allele] = (offset, kmer)

        # Second pass: keep only regions with a single allele match
        for region_id, alleles in region_to_alleles.items():
            if len(alleles) == 1:
                allele = next(iter(alleles))
                offset, kmer = region_kmer_hits[region_id][allele]
                allele_region_offsets.setdefault(motif_pos, {}).setdefault(allele, {})[region_id] = offset
                allele_matched_kmers.setdefault(motif_pos, {}).setdefault(allele, []).append(kmer)

    if debug:
        sys.stderr.write(f"\n=== [match_snv_aligned_kmers] DEBUG SUMMARY ===\n")
        sys.stderr.write(f"SNV: {snv_info['chrom']}:{snv_info['pos']} {snv_info['ref']}>{snv_info['alt']}\n")
        sys.stderr.write(f"Region: {snv_info['region_start']}-{snv_info['region_end']} ({snv_info['region_seq']})\n")
        sys.stderr.write(f"Number of wildcard kmers generated: {len(wildcards)}\n")

        total_hits = 0
        for motif_pos, wildcard in wildcards:
            if motif_pos not in allele_region_offsets:
                continue
            sys.stderr.write(f"\nMotif position {motif_pos} — Wildcard: {wildcard}\n")
            for allele, regions in allele_region_offsets[motif_pos].items():
                n_regions = len(regions)
                matched_kmers = sorted(set(allele_matched_kmers[motif_pos][allele]))
                total_hits += n_regions
                sys.stderr.write(f"  Allele {allele}: {n_regions} region(s), {len(matched_kmers)} unique k-mer(s)\n")
                for k in matched_kmers:
                    sys.stderr.write(f"    ↳ {k}\n")

        sys.stderr.write(f"\nTotal matched regions (deduplicated): {total_hits}\n")
        sys.stderr.write(f"===============================================\n")

    return allele_region_offsets, allele_matched_kmers


allele_region_offsets, allele_matched_kmers = match_snv_aligned_kmers(snv_info, kmers, kmer_size, debug = True)


=== [match_snv_aligned_kmers] DEBUG SUMMARY ===
SNV: chr5:1295113 C>T
Region: 1295106-1295120 (CAGCCCCCTCCGGGC)
Number of wildcard kmers generated: 8

Motif position 0 — Wildcard: CAGCCCC.
  Allele A: 4937 region(s), 1 unique k-mer(s)
    ↳ CAGCCCCA
  Allele C: 2997 region(s), 1 unique k-mer(s)
    ↳ CAGCCCCC
  Allele G: 2806 region(s), 1 unique k-mer(s)
    ↳ CAGCCCCG
  Allele T: 3874 region(s), 1 unique k-mer(s)
    ↳ CAGCCCCT

Motif position 1 — Wildcard: AGCCCC.T
  Allele A: 1867 region(s), 1 unique k-mer(s)
    ↳ AGCCCCAT
  Allele C: 2091 region(s), 1 unique k-mer(s)
    ↳ AGCCCCCT
  Allele G: 881 region(s), 1 unique k-mer(s)
    ↳ AGCCCCGT
  Allele T: 1808 region(s), 1 unique k-mer(s)
    ↳ AGCCCCTT

Motif position 2 — Wildcard: GCCCC.TC
  Allele A: 1992 region(s), 1 unique k-mer(s)
    ↳ GCCCCATC
  Allele C: 3557 region(s), 1 unique k-mer(s)
    ↳ GCCCCCTC
  Allele G: 1058 region(s), 1 unique k-mer(s)
    ↳ GCCCCGTC
  Allele T: 2574 region(s), 1 unique k-mer(s)
    ↳ GCCCCTTC



In [26]:
import math
import pandas as pd
import statsmodels.api as sm

def build_allele_matrix(probe_intensities, allele_region_offsets, exclude_alleles=None):
    """
    Builds the binary matrix X (probes × alleles) and intensity vector y.

    Returns:
        X: np.ndarray (N x A)
        y: np.ndarray (N,)
        allele_list: list of alleles (column order)
        used_regions: list of regions (row order)
    """
    if exclude_alleles is None:
        exclude_alleles = set()

    X = []
    y = []
    used_regions = []
    region_set = set()

    # Get full list of alleles (columns)
    allele_list = sorted({
        allele
        for pos in allele_region_offsets.values()
        for allele in pos
        if allele not in exclude_alleles
    })

    # Build region-to-alleles lookup
    region_to_alleles = {}
    for pos_dict in allele_region_offsets.values():
        for allele, region_dict in pos_dict.items():
            if allele in exclude_alleles:
                continue
            for region in region_dict:
                region_to_alleles.setdefault(region, set()).add(allele)

    # Build X and y matrices
    for region, alleles_present in region_to_alleles.items():
        if region not in probe_intensities or region in region_set:
            continue
        region_set.add(region)
        used_regions.append(region)
        row = [1 if allele in alleles_present else 0 for allele in allele_list]
        X.append(row)
        y.append(probe_intensities[region])

    return np.array(X), np.array(y), allele_list, used_regions



def run_per_motif_regression(
    allele_region_offsets,
    probe_intensities,
    snv_info,
    model_type="neg-binomial",  # or "ols"
    include_covariates=True,
    debug=False
):
    results = []

    for motif_pos, allele_dict in allele_region_offsets.items():
        if debug:
            sys.stderr.write(f"\n=== Running regression for motif position {motif_pos} ===\n")

        # Build allele matrix (excluding ref allele)
        X, y, alleles, used_regions  = build_allele_matrix(
            probe_intensities,
            {motif_pos: allele_dict},
            exclude_alleles={snv_info["ref"]}
        )

        if len(y) < 10 or X.shape[1] == 0:
            if debug:
                sys.stderr.write("  [!] Not enough data. Skipping.\n")
            continue

        X = pd.DataFrame(X, columns=alleles)

        if include_covariates:
            # Build region → offset dict for current motif
            region_to_offset = {
                region: offset
                for allele_regions in allele_dict.values()
                for region, offset in allele_regions.items()
                if region in probe_intensities
            }

            lp, sl = extract_covariates(used_regions, region_to_offset)

            X["lp"] = lp
            X["sl"] = sl

        X_const = sm.add_constant(X)

        if model_type == "ols":
            y_transformed = np.log1p(y)
            model = sm.OLS(y_transformed, X_const)
        else:
            model = sm.GLM(y, X_const, family=sm.families.NegativeBinomial())

        fit = model.fit()

        if debug:
            sys.stderr.write(f"  Shape of X: {X.shape}\n")
            sys.stderr.write(f"  Alleles tested: {alleles}\n")
            sys.stderr.write(f"  Coefs: {fit.params.tolist()}\n")
            sys.stderr.write(f"  P-values: {fit.pvalues.tolist()}\n")

        for i, allele in enumerate(alleles):
            results.append({
                "snv_str" : snv_info["snv_str"],
                "motif_pos": motif_pos,
                "allele": allele,
                "coef": fit.params[i + 1],  # +1 because of intercept
                "pval": fit.pvalues[i + 1],
                "n": int((X[allele] == 1).sum())
            })

    return results

results = run_per_motif_regression(allele_region_offsets, intensities, snv_info, include_covariates=True)
results

[{'snv_str': 'chr5:1295113:C>T',
  'motif_pos': 0,
  'allele': 'A',
  'coef': -0.023516832657355276,
  'pval': 0.11911048920257454,
  'n': 4937},
 {'snv_str': 'chr5:1295113:C>T',
  'motif_pos': 0,
  'allele': 'G',
  'coef': 0.28963371996627374,
  'pval': 9.064118093415818e-52,
  'n': 2806},
 {'snv_str': 'chr5:1295113:C>T',
  'motif_pos': 0,
  'allele': 'T',
  'coef': 0.08450358484319295,
  'pval': 2.1057119381378105e-07,
  'n': 3874},
 {'snv_str': 'chr5:1295113:C>T',
  'motif_pos': 1,
  'allele': 'A',
  'coef': -0.026903032275276774,
  'pval': 0.27247983387213437,
  'n': 1867},
 {'snv_str': 'chr5:1295113:C>T',
  'motif_pos': 1,
  'allele': 'G',
  'coef': 0.24040318674733077,
  'pval': 9.478860901243304e-14,
  'n': 881},
 {'snv_str': 'chr5:1295113:C>T',
  'motif_pos': 1,
  'allele': 'T',
  'coef': 0.07903944449821029,
  'pval': 0.0017032556035683774,
  'n': 1808},
 {'snv_str': 'chr5:1295113:C>T',
  'motif_pos': 2,
  'allele': 'A',
  'coef': -0.039786121787863116,
  'pval': 0.08135566139

In [23]:
import pandas as pd
import math

def print_rows_as_tsv_scan_style_with_snv(
    rows,
    header=(
        "snv",
        "wildcard_kmer",
        "filled_kmer",
        "window_index",
        "snp_index",
        "type",
        "allele",
        "coef",
        "pval",
    ),
):
    """
    Print regression results in scan-style TSV format with SNV string as first column.
    Replaces NaNs with the string "NaN".
    """
    df_rows = []

    for row in rows:
        # snv_info["wildcards"] is needed to fill in the kmer fields
        snv_str = row["snv_str"]
        motif_pos = row["motif_pos"]
        allele = row["allele"]

        wildcard_kmer, filled_kmers = row["wildcard"]
        filled_kmer = filled_kmers.get(allele, "NaN")

        formatted = [
            snv_str,
            wildcard_kmer,
            filled_kmer,
            motif_pos,
            row.get("snp_index", "NaN"),
            "AFF",
            allele,
            row["coef"],
            row["pval"],
        ]
        df_rows.append([
            "NaN" if (isinstance(x, float) and math.isnan(x)) or x == "" else x
            for x in formatted
        ])

    df = pd.DataFrame(df_rows, columns=header)
    print(df.to_csv(sep="\t", index=False))





# === Convert regression results into scan-style rows ===

scan_rows = []

for result in results:  # Replace with your actual results variable
    motif_pos = result["motif_pos"]
    allele = result["allele"]
    coef = result["coef"]
    pval = result["pval"]

    # Pull from snv_info for context
    snv_index = snv_info["snv_index"] if "snv_index" in snv_info else snv_info["snv_index"]
    region_seq = snv_info["region_seq"]
    wildcard_kmer = region_seq[motif_pos : motif_pos + kmer_size]
    filled_kmer = list(wildcard_kmer)
    filled_kmer[snv_index - motif_pos] = allele if 0 <= snv_index - motif_pos < kmer_size else "."
    filled_kmer = "".join(filled_kmer)
    wildcard_kmer = wildcard_kmer[: snv_index - motif_pos] + "." + wildcard_kmer[snv_index - motif_pos + 1:]

    scan_rows.append([
        wildcard_kmer,
        filled_kmer,
        motif_pos,
        snv_index,
        "AFF",
        allele,
        coef,
        pval,
        snv_info["pos"],  # not printed in scan version, but could keep for full version
    ])

# === Print final TSV output in scan style ===
print_rows_as_tsv_scan_style_with_snv(scan_rows)


KeyError: 'motif_pos'